<a href="https://colab.research.google.com/github/Prktriesitout/AI-Coach/blob/main/data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

# Load the dataset
file_path = '/content/split_part_8.xlsx'
df = pd.read_excel(file_path)

# Display basic information and first few rows
print("Original dataset shape:", df.shape)
display(df.head())


Original dataset shape: (5518, 17)


,CONSUMER_ID,SECTION_NAME,CATEGORY,CONSUMER_NAME,ADDRESS,MOBILE_NUMBER,METER_TYPE,CONNECTION_DEAMND,PHASE,CONNECTION_STATUS,METER_NUMBER,DISCONNECTION_DATE,RECONNECTION_DATE,DUE_DATE,CURRENT_READING,READING_STATUS,DUES
0,234208493150,JAGDISHPUR,DS1D,MDINTEKHAR ALAM,"NAYA TOLA PURANI , NAYA TOLA PURANINAYA TOLA P...",7.979928e+09,NORMAL,1,1,RUNNING,S1460369,10-MAR-2025,11-MAR-2025,09-FEB-2026,8139,OK,3406.38
1,234210927239,JAGDISHPUR,DS1D,AMIT KUMAR,"FLAT NO 9, ROAD NO 6VASTU VIHAR PHASE 1, SIMPL...",9.470363e+09,SMART_PREPAID,1,1,RUNNING,7749556,NaN,NaN,22-JAN-2026,2149,OK,-145.38
2,234207135314,JAGDISHPUR,DS1D,ASHOK KUMAR SHARMA,"VILL-PISTA ,TOLA-PISTAPANCH-BAIJANI ,BLOCK-JAG...",8.076368e+09,SMART_PREPAID,1,1,RUNNING,7484957,NaN,NaN,22-JAN-2026,2035,OK,-1540.86
3,234207134598,JAGDISHPUR,DS1D,PRADEEP KUMAR SINGH,"VILL-PISTA ,TOLA-PISTAPANCH-BAIJANI ,BLOCK-JAG...",8.199950e+09,SMART_PREPAID,2,1,RUNNING,7115194,NaN,NaN,22-JAN-2026,3772,OK,-116.70
4,234207134878,JAGDISHPUR,DS1D,JANTU PASWAN,"VILL-PISTA ,TOLA-PISTAPANCH-BAIJANI ,BLOCK-JAG...",9.142532e+09,SMART_PREPAID,1,1,RUNNING,7123530,02-JAN-2023,22-MAY-2023,22-JAN-2026,153,OK,2973.15


In [ ]:
# 1. Clean data: Drop rows where MOBILE_NUMBER or METER_NUMBER is missing or empty
df_clean = df.dropna(subset=['MOBILE_NUMBER', 'METER_NUMBER']).copy()
# Also filter out any rows where they might be blank strings
df_clean = df_clean[df_clean['MOBILE_NUMBER'].astype(str).str.strip() != '']
df_clean = df_clean[df_clean['METER_NUMBER'].astype(str).str.strip() != '']

print(f"Dropped records missing phone or meter number. New shape: {df_clean.shape}")

# 2. Applying Rules

# Rule 1: Consistently 'OK' Reading Status
# In this dataset, the status is indicated in the 'READING_STATUS' column.
df_clean['is_consistently_ok'] = df_clean['READING_STATUS'].astype(str).str.strip().str.upper() == 'OK'

# Rule 2: High Consumption + Postpaid Meter + Zero Late Days
# - High Consumption: We use 'CURRENT_READING' as the indicator for cumulative consumption. Let's flag the top 20% (80th percentile).
high_cons_threshold = df_clean['CURRENT_READING'].quantile(0.80)
df_clean['high_consumption'] = df_clean['CURRENT_READING'] >= high_cons_threshold

# - Postpaid Meter: Postpaid is represented as 'NORMAL' in 'METER_TYPE' (vs 'SMART_PREPAID')
df_clean['is_postpaid'] = df_clean['METER_TYPE'].astype(str).str.strip().str.upper() == 'NORMAL'

# - Zero Late Days: Since there's no explicitly named 'LATE_DAYS' column,
# we'll look at whether the account has negative or zero dues to imply no overdue balance, or base it on absence of disconnection history.
# For the rule definition '0 Days Past Due', we will proxy it where DUES <= 0.
df_clean['zero_late_days'] = df_clean['DUES'] <= 0

# Combine for Feature 2
df_clean['high_value_postpaid'] = (
    df_clean['high_consumption'] &
    df_clean['is_postpaid'] &
    df_clean['zero_late_days']
)

# Rule 3: Negative Dues (Credit Balances)
df_clean['has_negative_dues'] = df_clean['DUES'] < 0

# Create the final target flags & filter results
print("\nSummary of targeted segments found:")
print(df_clean[['is_consistently_ok', 'high_value_postpaid', 'has_negative_dues']].sum())

# Save the final structured dataset
df_clean.to_excel('/content/processed_dataset.xlsx', index=False)
print("\nSaved the final dataset to /content/processed_dataset.xlsx")

# Display first few matches of negative dues and consistently OK records
display(df_clean[df_clean['has_negative_dues'] | df_clean['high_value_postpaid']].head())


Dropped records missing phone or meter number. New shape: (5174, 17)

Summary of targeted segments found:
is_consistently_ok     4588
high_value_postpaid     242
has_negative_dues      2870
dtype: int64

Saved the final dataset to /content/processed_dataset.xlsx


,CONSUMER_ID,SECTION_NAME,CATEGORY,CONSUMER_NAME,ADDRESS,MOBILE_NUMBER,METER_TYPE,CONNECTION_DEAMND,PHASE,CONNECTION_STATUS,...,DUE_DATE,CURRENT_READING,READING_STATUS,DUES,is_consistently_ok,high_consumption,is_postpaid,zero_late_days,high_value_postpaid,has_negative_dues
1,234210927239,JAGDISHPUR,DS1D,AMIT KUMAR,"FLAT NO 9, ROAD NO 6VASTU VIHAR PHASE 1, SIMPL...",9.470363e+09,SMART_PREPAID,1,1,RUNNING,...,22-JAN-2026,2149,OK,-145.38,True,False,False,True,False,True
2,234207135314,JAGDISHPUR,DS1D,ASHOK KUMAR SHARMA,"VILL-PISTA ,TOLA-PISTAPANCH-BAIJANI ,BLOCK-JAG...",8.076368e+09,SMART_PREPAID,1,1,RUNNING,...,22-JAN-2026,2035,OK,-1540.86,True,False,False,True,False,True
3,234207134598,JAGDISHPUR,DS1D,PRADEEP KUMAR SINGH,"VILL-PISTA ,TOLA-PISTAPANCH-BAIJANI ,BLOCK-JAG...",8.199950e+09,SMART_PREPAID,2,1,RUNNING,...,22-JAN-2026,3772,OK,-116.70,True,False,False,True,False,True
5,234207135050,JAGDISHPUR,DS1D,SEEMA KUMARI,"VILL-JAGDISHPUR ,TOLA-JAGDISHPURPANCH-JAGADISH...",9.398008e+09,SMART_PREPAID,3,1,RUNNING,...,23-JAN-2026,342,OK,-207.45,True,False,False,True,False,True
6,234207134932,JAGDISHPUR,DS1D,DIVYA KUMARI,"VILL-JAGDISHPUR ,TOLA-JAGDISHPURPANCH-JAGADISH...",7.683063e+09,SMART_PREPAID,1,1,RUNNING,...,22-JAN-2026,947,OK,-121.39,True,False,False,True,False,True


In [ ]:
import pandas as pd
import numpy as np

# 1. Clean data: Drop rows where phone number or meter number is missing
df_clean = df.dropna(subset=['MOBILE_NUMBER', 'METER_NUMBER']).copy()
df_clean = df_clean[df_clean['MOBILE_NUMBER'].astype(str).str.strip() != '']
df_clean = df_clean[df_clean['METER_NUMBER'].astype(str).str.strip() != '']

# Standardize address and clean names for comparisons
df_clean['CLEAN_ADDRESS'] = df_clean['ADDRESS'].astype(str).str.strip().str.upper()
df_clean['CLEAN_NAME'] = df_clean['CONSUMER_NAME'].astype(str).str.strip().str.upper()

# Helper function to extract a potential last name/surname
def get_last_name(name):
    parts = name.split()
    return parts[-1] if len(parts) > 0 else ""

df_clean['SURNAME'] = df_clean['CLEAN_NAME'].apply(get_last_name)

# Group by standardized address
address_groups = df_clean.groupby('CLEAN_ADDRESS').filter(lambda x: len(x['METER_NUMBER'].unique()) in [2, 3])

# Filter groups where they either have the identical name or share a last name
def filter_shared_ownership(group):
    unique_names = group['CLEAN_NAME'].unique()
    unique_surnames = group['SURNAME'].unique()
    # Filter out empty surnames if any
    unique_surnames = [s for s in unique_surnames if s != ""]

    # Gold match: 1 unique full name (same owner) or exactly 1 unique surname (same family)
    if len(unique_names) == 1 or len(unique_surnames) == 1:
        return True
    return False

# Apply the grouping and matching filter
multimeter_owners = address_groups.groupby('CLEAN_ADDRESS').filter(filter_shared_ownership).copy()

print(f"Found {multimeter_owners['CLEAN_ADDRESS'].nunique()} unique locations with 2 or 3 meters owned by the same individual/family.")
print(f"Total associated records matching this pattern: {multimeter_owners.shape[0]}")

# Save the final filtered dataset
multimeter_owners.to_excel('/content/multimeter_filtered_dataset.xlsx', index=False)
print("Saved the final multimeter dataset to /content/multimeter_filtered_dataset.xlsx")

# Display the top matched properties
display(multimeter_owners[['CONSUMER_NAME', 'ADDRESS', 'METER_NUMBER', 'CONNECTION_DEAMND', 'MOBILE_NUMBER']].head(10))


Found 15 unique locations with 2 or 3 meters owned by the same individual/family.
Total associated records matching this pattern: 33
Saved the final multimeter dataset to /content/multimeter_filtered_dataset.xlsx


,CONSUMER_NAME,ADDRESS,METER_NUMBER,CONNECTION_DEAMND,MOBILE_NUMBER
79,SURENDRA PD SINGH,"PANCH-JAGADISHPUR, VILL-JAGDISHPUR, TOLA-JAGDI...",7202957,1,8.298119e+09
137,RAJENDRA PRASAD SINGH,"PANCH-JAGADISHPUR, VILL-JAGDISHPUR, TOLA-JAGDI...",7202960,1,9.534330e+09
203,VINAY DAS,"PANCH-JAGADISHPUR, VILL-JAGDISHPUR, TOLA-JAGDI...",7203541,1,8.102091e+09
272,MURARI DAS,"PANCH-JAGADISHPUR, VILL-JAGDISHPUR, TOLA-JAGDI...",7893105,1,7.281978e+09
747,PUTUL DEVI,"12, BIHARFATEHPUR , JAGDISHPURJAGDISHPUR , PIN...",7940992,1,7.061696e+09
826,RUBI DEVI,"12, BIHARFATEHPUR , JAGDISHPURJAGDISHPUR , PIN...",SB8331609,1,8.210809e+09
979,GOPAL SINGH,"PANCH-JAGADISHPUR, VILL-JAGDISHPUR, TOLA-JAGDI...",SB6788608,1,9.572282e+09
1011,GARIB DAS,"PANCH-JAGADISHPUR, VILL-JAGDISHPUR, TOLA-JAGDI...",7195445,1,7.250868e+09
1168,MADHURI DEVI,"PANCH-JAGADISHPUR, VILL-JAGDISHPUR, TOLA-JAGDI...",7875006,1,7.033974e+09
1173,SANGITA DEVI,"PANCH-JAGADISHPUR, VILL-JAGDISHPUR, TOLA-JAGDI...",7587822,1,8.271525e+09


In [ ]:
# Create a clean, final dataset containing only the targeted people with 2-3 connection demands
final_targets = multimeter_owners.copy()

# Drop our temporary cleaning columns to keep the file neat
columns_to_drop = ['CLEAN_ADDRESS', 'CLEAN_NAME', 'SURNAME']
final_targets = final_targets.drop(columns=[col for col in columns_to_drop if col in final_targets.columns])

# Sort by address and name for easy readability
final_targets = final_targets.sort_values(by=['ADDRESS', 'CONSUMER_NAME'])

# Save to excel file
output_path = '/content/final_multimeter_leads.xlsx'
final_targets.to_excel(output_path, index=False)

print(f"Successfully exported {final_targets.shape[0]} matching customer records to {output_path}!")
display(final_targets.head(10))


Successfully exported 33 matching customer records to /content/final_multimeter_leads.xlsx!


,CONSUMER_ID,SECTION_NAME,CATEGORY,CONSUMER_NAME,ADDRESS,MOBILE_NUMBER,METER_TYPE,CONNECTION_DEAMND,PHASE,CONNECTION_STATUS,METER_NUMBER,DISCONNECTION_DATE,RECONNECTION_DATE,DUE_DATE,CURRENT_READING,READING_STATUS,DUES
3070,234209531415,JAGDISHPUR,DS1D,MAYA DEVI,", SONUDIH SAINOSONUDIH SAINO, PO SAINOPS GORAD...",7.667707e+09,NORMAL,1,1,RUNNING,C125340,NaN,NaN,10-FEB-2026,9094,OK,2946.28
2938,234209531184,JAGDISHPUR,DS1D,MINA DEVI,", SONUDIH SAINOSONUDIH SAINO, PO SAINOPS GORAD...",7.667707e+09,NORMAL,1,1,RUNNING,C139325,NaN,NaN,10-FEB-2026,2758,OK,5578.60
747,234208799341,JAGDISHPUR,DS1D,PUTUL DEVI,"12, BIHARFATEHPUR , JAGDISHPURJAGDISHPUR , PIN...",7.061696e+09,SMART_PREPAID,1,1,RUNNING,7940992,NaN,NaN,22-JAN-2026,261,OK,2735.77
826,234208799435,JAGDISHPUR,DS1D,RUBI DEVI,"12, BIHARFATEHPUR , JAGDISHPURJAGDISHPUR , PIN...",8.210809e+09,NORMAL,1,1,RUNNING,SB8331609,NaN,NaN,10-FEB-2026,877,LK,4328.60
1965,234211923914,JAGDISHPUR,DS1D,PARWATI DEVI,"JAMNI, JAMNI,, PIN - 812005",7.050800e+09,SMART_PREPAID,1,1,RUNNING,7439044,NaN,NaN,22-JAN-2026,140,OK,-35.00
3510,234211890119,JAGDISHPUR,DS1D,SULEKHA DEVI,"JAMNI, JAMNI,, PIN - 812005",6.299358e+09,SMART_PREPAID,1,1,RUNNING,7436678,NaN,NaN,23-JAN-2026,438,OK,-98.00
2404,234208846000,JAGDISHPUR,DS1D,RINA DEVI,"NA, SOUNIDHSOUNIDH , SAINOGORADIH , PIN - 813105",8.862909e+09,SMART_PREPAID,1,1,RUNNING,7946083,NaN,NaN,22-JAN-2026,1675,OK,4578.82
2543,234209530496,JAGDISHPUR,DS1D,RITA DEVI,"NA, SOUNIDHSOUNIDH , SAINOGORADIH , PIN - 813105",6.287107e+09,SMART_PREPAID,1,1,RUNNING,7587592,NaN,NaN,23-JAN-2026,473,OK,5001.23
4642,23420054012,SABOUR,DS1D,RAJENDRA YADAV,"PANCH-BISHUNPUR JICHHO, VILL-BISHUNPUR JICHHO,...",8.271386e+09,SMART_PREPAID,1,1,RUNNING,7889537,NaN,NaN,22-JAN-2026,2050,OK,-342.72
4652,23420054080,SABOUR,DS1D,SANJEEV KUMAR YADAV,"PANCH-BISHUNPUR JICHHO, VILL-BISHUNPUR JICHHO,...",9.939386e+09,NORMAL,1,1,RUNNING,1419782,NaN,NaN,08-FEB-2026,3622,OK,23603.94


In [ ]:
import pandas as pd

# Load the original dataset
df_original = pd.read_excel('/content/split_part_8.xlsx')

# Filter strictly for connection demands having values of 2 or 3
# (And still ensuring phone and meter numbers exist as per original general instructions)
df_filtered = df_original[
    df_original['CONNECTION_DEAMND'].isin([2, 3]) &
    df_original['MOBILE_NUMBER'].notna() &
    df_original['METER_NUMBER'].notna()
].copy()

print(f"Found {df_filtered.shape[0]} records with Connection Demand of 2 or 3.")

# Save the simplified dataset
output_file = '/content/connection_demand_2_3_leads.xlsx'
df_filtered.to_excel(output_file, index=False)
print(f"Saved the filtered dataset to {output_file}")

# Display preview
display(df_filtered[['CONSUMER_NAME', 'ADDRESS', 'CONNECTION_DEAMND', 'MOBILE_NUMBER', 'METER_NUMBER']].head(10))


Found 360 records with Connection Demand of 2 or 3.
Saved the filtered dataset to /content/connection_demand_2_3_leads.xlsx


,CONSUMER_NAME,ADDRESS,CONNECTION_DEAMND,MOBILE_NUMBER,METER_NUMBER
3,PRADEEP KUMAR SINGH,"VILL-PISTA ,TOLA-PISTAPANCH-BAIJANI ,BLOCK-JAG...",2,8.199950e+09,7115194
5,SEEMA KUMARI,"VILL-JAGDISHPUR ,TOLA-JAGDISHPURPANCH-JAGADISH...",3,9.398008e+09,7855646
30,VINAY SINGH,"VILL-MUKHARIA ,TOLA-MUKHARIAPANCH-KHIRIBANDH ,...",3,7.992340e+09,7199248
32,DR.PADMESHWAR MISHRA,"VILL-PISTA ,TOLA-PISTAPANCH-BAIJANI ,BLOCK-JAG...",3,8.709817e+09,7586215
36,KEDARNATH MANDAL,"PANCH-BHAWANIPUR DESARI, VILL-BHAWANIPUR DERSI...",2,8.294801e+09,7857546
38,SUNAINA DEVI,"VILL-MUKHARIA ,TOLA-MUKHARIAPANCH-KHIRIBANDH ,...",2,9.431011e+09,7082704
54,VIJAY KUMAR SINGH,"192, JAGDISHPURIN FRONT OF MAHAVIR MANDIR , JA...",3,8.051605e+09,9053086
81,NIRANJAN PRASAD BHAGAT,"VILL-JAGDISHPUR ,TOLA-JAGDISHPURPANCH-JAGADISH...",2,8.757668e+09,7856583
85,KUMAR NITYANAND,"PANCH-JAGADISHPUR, VILL-JAGDISHPUR, TOLA-JAGDI...",2,9.955880e+09,7113039
114,ABHISHEK KUMAR,"PANCH-JAGADISHPUR, VILL-JAGDISHPUR, TOLA-JAGDI...",3,9.709596e+09,7113037
